# Latent-Observable Correlations

This notebook loads a trained single-channel no-PID CNN-VAE checkpoint, runs deterministic encoder inference on a reproducible event sample, identifies active latent features with the corrected signal-to-noise ratio definition, and correlates only those active posterior means plus one representative posterior standard deviation with image-based jet observables.

The observable calculations use the same pT-normalized event images seen by the VAE before optional model-input standardization. Decoder reconstructions are not used.

## Imports and Project Setup

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import os
import sys
import warnings
from typing import Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else (lambda x: x)

try:
    from numba import njit
    NUMBA_AVAILABLE = True
except Exception:
    NUMBA_AVAILABLE = False
    njit = None

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
CODE_DIR = PROJECT_ROOT / "code_training"
sys.path.insert(0, str(CODE_DIR))

from betaVae import BetaVAE
from jet_image_utils import (
    load_events_from_dat,
    event_to_jet_image,
    event_to_jet_image_no_pid,
    pt_normalize,
    build_pid_channel_map,
    build_pid_lookup,
)

plt.rcParams["figure.dpi"] = 120
torch.set_grad_enabled(False)
PROJECT_ROOT


## Configuration

In [ ]:
CKPT_PATH = PROJECT_ROOT / "results" / "200GeVjet_nopid_annealing/mse_all" / "beta_0p000001" / "best_model_betavae.pth"
DATA_PATH = PROJECT_ROOT.parent / "data" / "u_jet_real_sample.dat"
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "data" / "u_jet_real_sample.dat"

N_ANALYSIS_EVENTS = None
BATCH_SIZE = 256
RANDOM_SEED = 0

JET_RADIUS = 0.5
ETA_MAX = 0.5
PHI_MAX = 0.5

SNR_THRESHOLD = 1.0

# None means automatically select a representative sigma.
# An integer forces the notebook to use sigma from that latent dimension.
SIGMA_FEATURE_INDEX = None

PIXEL_ACTIVE_THRESHOLD = 1e-12

RHO_EPS = 1e-12
D2_EPS = 1e-12
TAU_EPS = 1e-12
COVARIANCE_EPS = 1e-15

ANGLE_ECCENTRICITY_CUT = 0.05

OUTPUT_DIR = NOTEBOOK_DIR / "latent_observable_outputs"

OBSERVABLE_IMPLEMENTATION_VERSION = "latent_observable_v1"
TAU_AXIS_PRESCRIPTION = "deterministic weighted-kmeans image-based tau21 approximation"
N_DISPLAY_SCATTER_EXAMPLES = 6

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"
assert DATA_PATH.exists(), f"Data file not found: {DATA_PATH}"
print("checkpoint:", CKPT_PATH)
print("data      :", DATA_PATH)
print("device    :", DEVICE)


## Helper Functions Copied or Adapted from `compare_reconstruction_images.ipynb`

In [ ]:
def pid_map_from_checkpoint_or_events(checkpoint: dict, events: list[np.ndarray]) -> dict[int, int]:
    if "pid_channel_map" in checkpoint:
        return {int(pid): int(ch) for pid, ch in checkpoint["pid_channel_map"].items()}
    if int(checkpoint.get("n_channels", 1)) == 1 and checkpoint_channel_mode(checkpoint) == "eta_phi_total_pt_no_pid":
        return {0: 0}
    return build_pid_channel_map(events)

def checkpoint_hidden_dims(checkpoint: dict) -> list[int]:
    return list(checkpoint.get("hidden_dims", [32, 64, 128, 256, 512]))

def checkpoint_output_mode(checkpoint: dict) -> str:
    if "output_mode" in checkpoint:
        return str(checkpoint["output_mode"])
    if checkpoint.get("normalization", None) == "event_pt":
        return "event_pt"
    return "tanh"

def checkpoint_channel_mode(checkpoint: dict) -> str:
    if "channel_mode" in checkpoint:
        return str(checkpoint["channel_mode"])
    pid_map = checkpoint.get("pid_channel_map", {})
    pid_map = {int(pid): int(ch) for pid, ch in pid_map.items()} if hasattr(pid_map, "items") else {}
    if int(checkpoint["n_channels"]) == 1 and (not pid_map or pid_map == {0: 0}):
        return "eta_phi_total_pt_no_pid"
    return "pid_channels"

def checkpoint_uses_no_pid_images(checkpoint: dict) -> bool:
    return checkpoint_channel_mode(checkpoint) == "eta_phi_total_pt_no_pid"

def checkpoint_uses_standardized_images(checkpoint: dict) -> bool:
    return "mean_img" in checkpoint and "std_img" in checkpoint and checkpoint.get("normalization", None) != "event_pt"

def load_model(checkpoint: dict, device: torch.device) -> BetaVAE:
    model = BetaVAE(
        in_channels=int(checkpoint["n_channels"]),
        latent_dim=int(checkpoint["latent_dim"]),
        image_size=int(checkpoint["img_size"]),
        hidden_dims=checkpoint_hidden_dims(checkpoint),
        output_mode=checkpoint_output_mode(checkpoint),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model

def make_event_image(event: np.ndarray, pid_lookup: np.ndarray, pid_min: int, n_channels: int, img_size: int, checkpoint: dict) -> np.ndarray:
    if checkpoint_uses_no_pid_images(checkpoint):
        img = event_to_jet_image_no_pid(
            event,
            eta_bins=img_size,
            phi_bins=img_size,
            eta_max=ETA_MAX,
            phi_max=PHI_MAX,
        )
    else:
        img = event_to_jet_image(
            event,
            pid_lookup,
            pid_min,
            n_channels,
            eta_bins=img_size,
            phi_bins=img_size,
            eta_max=ETA_MAX,
            phi_max=PHI_MAX,
        )
    return pt_normalize(img).astype(np.float32)

def model_input_from_event_image(img: np.ndarray, checkpoint: dict) -> np.ndarray:
    if checkpoint_uses_standardized_images(checkpoint):
        mean_img = np.asarray(checkpoint["mean_img"], dtype=np.float32)
        std_img = np.asarray(checkpoint["std_img"], dtype=np.float32)
        return ((img - mean_img) / (std_img + 1e-5)).astype(np.float32)
    return img.astype(np.float32)

def file_mtime(path: Path) -> float | None:
    try:
        return path.stat().st_mtime
    except FileNotFoundError:
        return None

def json_ready(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    return value


## Load Checkpoint, Model, and Events

In [ ]:
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
events = load_events_from_dat(str(DATA_PATH))
pid_channel_map = pid_map_from_checkpoint_or_events(checkpoint, events)
pid_lookup, pid_min = build_pid_lookup(pid_channel_map)
model = load_model(checkpoint, DEVICE)

img_size = int(checkpoint["img_size"])
n_channels = int(checkpoint["n_channels"])
latent_dim = int(checkpoint["latent_dim"])
channel_mode = checkpoint_channel_mode(checkpoint)
normalization_mode = str(checkpoint.get("normalization", "standardized" if checkpoint_uses_standardized_images(checkpoint) else "event_pt"))

print("checkpoint path :", CKPT_PATH)
print("data path       :", DATA_PATH)
print("device          :", DEVICE)
print("loaded events   :", len(events))
print("img_size        :", img_size)
print("n_channels      :", n_channels)
print("latent_dim      :", latent_dim)
print("hidden_dims     :", checkpoint_hidden_dims(checkpoint))
print("output_mode     :", checkpoint_output_mode(checkpoint))
print("channel_mode    :", channel_mode)
print("normalization   :", normalization_mode)
print("val_loss        :", checkpoint.get("val_loss", checkpoint.get("best_val_loss")))

if not checkpoint_uses_no_pid_images(checkpoint):
    raise ValueError(f"This analysis is restricted to no-PID single-channel jet images, got channel_mode={channel_mode!r}.")
if n_channels != 1:
    raise ValueError(f"This analysis is restricted to one image channel, got n_channels={n_channels}.")


## Inspect Image Convention and Pixel Coordinates

In [ ]:
# In code_training/jet_image_utils.py, event_to_jet_image_no_pid computes:
#   flat_idx = h_bins * eta_bins + w_bins
#   image = bincount(...).reshape(1, phi_bins, eta_bins)
# Therefore image[0, phi_index, eta_index] is the pT at that phi-eta pixel.
eta_centers = -ETA_MAX + (np.arange(img_size) + 0.5) * (2.0 * ETA_MAX / img_size)
phi_centers = -PHI_MAX + (np.arange(img_size) + 0.5) * (2.0 * PHI_MAX / img_size)
ETA_GRID, PHI_GRID = np.meshgrid(eta_centers, phi_centers, indexing="xy")

assert ETA_GRID.shape == (img_size, img_size)
assert PHI_GRID.shape == (img_size, img_size)
print("image spatial axes: image[0, phi_index, eta_index]")
print("ETA_GRID shape:", ETA_GRID.shape, "PHI_GRID shape:", PHI_GRID.shape)
print("eta range:", (float(eta_centers.min()), float(eta_centers.max())))
print("phi range:", (float(phi_centers.min()), float(phi_centers.max())))

diagnostic_img = make_event_image(events[0], pid_lookup, pid_min, n_channels, img_size, checkpoint)[0]
max_phi_idx, max_eta_idx = np.unravel_index(np.argmax(diagnostic_img), diagnostic_img.shape)
fig, ax = plt.subplots(figsize=(5.2, 4.6), constrained_layout=True)
im = ax.imshow(
    diagnostic_img,
    origin="lower",
    extent=[-ETA_MAX, ETA_MAX, -PHI_MAX, PHI_MAX],
    aspect="equal",
)
ax.scatter([ETA_GRID[max_phi_idx, max_eta_idx]], [PHI_GRID[max_phi_idx, max_eta_idx]], c="red", s=40, label="max pixel center")
ax.set_xlabel("eta")
ax.set_ylabel("phi")
ax.set_title("Input image grid diagnostic")
ax.legend(loc="upper right")
fig.colorbar(im, ax=ax, label="pT fraction")
plt.show()


## Reproducible Event Selection

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
if N_ANALYSIS_EVENTS is None:
    analysis_event_indices = np.arange(len(events), dtype=np.int64)
else:
    if N_ANALYSIS_EVENTS <= 0:
        raise ValueError("N_ANALYSIS_EVENTS must be None or a positive integer.")
    if N_ANALYSIS_EVENTS > len(events):
        raise ValueError(f"N_ANALYSIS_EVENTS={N_ANALYSIS_EVENTS} exceeds loaded events={len(events)}.")
    analysis_event_indices = rng.choice(len(events), size=int(N_ANALYSIS_EVENTS), replace=False).astype(np.int64)

n_analyzed_requested = int(len(analysis_event_indices))
print("loaded events   :", len(events))
print("selected events :", n_analyzed_requested)
print("first indices   :", analysis_event_indices[:10].tolist())

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "tables").mkdir(exist_ok=True)
(OUTPUT_DIR / "heatmaps").mkdir(exist_ok=True)
(OUTPUT_DIR / "scatter_plots").mkdir(exist_ok=True)
(OUTPUT_DIR / "cache").mkdir(exist_ok=True)


## Batched `mu`, `logvar`, and `sigma` Inference

In [ ]:
cache_metadata = {
    "checkpoint_path": str(CKPT_PATH.resolve()),
    "checkpoint_mtime": file_mtime(CKPT_PATH),
    "data_path": str(DATA_PATH.resolve()),
    "data_mtime": file_mtime(DATA_PATH),
    "loaded_events": len(events),
    "selected_event_indices": analysis_event_indices.tolist(),
    "random_seed": RANDOM_SEED,
    "image_size": img_size,
    "n_channels": n_channels,
    "latent_dim": latent_dim,
    "eta_max": ETA_MAX,
    "phi_max": PHI_MAX,
    "jet_radius": JET_RADIUS,
    "snr_threshold": SNR_THRESHOLD,
    "pixel_active_threshold": PIXEL_ACTIVE_THRESHOLD,
    "rho_eps": RHO_EPS,
    "d2_eps": D2_EPS,
    "tau_eps": TAU_EPS,
    "covariance_eps": COVARIANCE_EPS,
    "angle_eccentricity_cut": ANGLE_ECCENTRICITY_CUT,
    "observable_implementation_version": OBSERVABLE_IMPLEMENTATION_VERSION,
}

latent_cache_npz = OUTPUT_DIR / "cache" / "latent_arrays.npz"
latent_cache_json = OUTPUT_DIR / "cache" / "latent_metadata.json"

def metadata_matches(path: Path, expected: dict) -> bool:
    if not path.exists():
        return False
    try:
        actual = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return False
    return actual == expected

if latent_cache_npz.exists() and metadata_matches(latent_cache_json, cache_metadata):
    cached = np.load(latent_cache_npz)
    latent_mu = cached["latent_mu"]
    latent_logvar = cached["latent_logvar"]
    print("Loaded latent arrays from cache:", latent_cache_npz)
else:
    latent_mu_batches = []
    latent_logvar_batches = []
    for start in tqdm(range(0, n_analyzed_requested, BATCH_SIZE), desc="Encoder inference"):
        batch_indices = analysis_event_indices[start:start + BATCH_SIZE]
        batch_images = np.stack([
            model_input_from_event_image(
                make_event_image(events[int(i)], pid_lookup, pid_min, n_channels, img_size, checkpoint),
                checkpoint,
            )
            for i in batch_indices
        ])
        x_batch = torch.from_numpy(batch_images).to(DEVICE)
        with torch.no_grad():
            mu_batch, logvar_batch = model.encode(x_batch)
        latent_mu_batches.append(mu_batch.cpu().numpy())
        latent_logvar_batches.append(logvar_batch.cpu().numpy())
    latent_mu = np.concatenate(latent_mu_batches, axis=0)
    latent_logvar = np.concatenate(latent_logvar_batches, axis=0)
    np.savez_compressed(latent_cache_npz, latent_mu=latent_mu, latent_logvar=latent_logvar)
    latent_cache_json.write_text(json.dumps(cache_metadata, indent=2, default=json_ready), encoding="utf-8")
    print("Saved latent arrays to cache:", latent_cache_npz)

latent_sigma = np.exp(0.5 * latent_logvar.astype(np.float64))
latent_variance = np.exp(latent_logvar.astype(np.float64))
print("latent_mu shape    :", latent_mu.shape)
print("latent_logvar shape:", latent_logvar.shape)
print("latent_sigma shape :", latent_sigma.shape)


## SNR Calculation and Active-Feature Selection

The active feature definition is corrected here to match the stated SNR formula:

`SNR_i = Var_x[mu_i(x)] / E_x[sigma_i^2(x)]`.

The denominator is the event-average posterior variance `mean(exp(logvar))`, not the across-event variance `var(exp(logvar))`. Active features use the strict condition `SNR > 1` by default.

In [ ]:
signal_variance = np.var(latent_mu.astype(np.float64), axis=0, ddof=0)
mean_posterior_variance = np.mean(np.exp(latent_logvar.astype(np.float64)), axis=0)
latent_snr = signal_variance / np.maximum(mean_posterior_variance, np.finfo(np.float64).tiny)
active_mask = latent_snr > SNR_THRESHOLD
active_feature_indices = np.flatnonzero(active_mask)
latent_features = np.arange(latent_dim)

snr_df = pd.DataFrame({
    "latent_index": latent_features,
    "signal_variance": signal_variance,
    "mean_posterior_variance": mean_posterior_variance,
    "snr": latent_snr,
    "is_active": active_mask,
})
snr_path = OUTPUT_DIR / "tables" / "latent_snr.csv"
snr_df.to_csv(snr_path, index=False)

bar_colors = np.where(active_mask, "tab:blue", "0.65")
fig, ax = plt.subplots(figsize=(13, 4.8), constrained_layout=True)
ax.bar(latent_features, latent_snr, color=bar_colors, edgecolor="black", linewidth=0.4)
ax.axhline(SNR_THRESHOLD, color="tab:red", linestyle="--", linewidth=1.2, label=f"SNR = {SNR_THRESHOLD:g}")
#ax.set_yscale("log")
ax.set_yscale("linear")
ax.set_xticks(latent_features)
ax.set_xlabel("latent feature index")
ax.set_ylabel(r"$Var_x[mu_i(x)] / E_x[sigma_i^2(x)]$ (log scale)")
ax.set_title(f"Latent signal-to-noise ratio; {active_mask.sum()}/{latent_dim} active features")
ax.grid(axis="y", which="both", alpha=0.25)
ax.legend()
fig.savefig(OUTPUT_DIR / "heatmaps" / "latent_snr.png", dpi=220)
fig.savefig(OUTPUT_DIR / "heatmaps" / "latent_snr.pdf")
plt.show()

print("active feature indices:", active_feature_indices.tolist())
print("SNR table saved to   :", snr_path)
if len(active_feature_indices) == 0:
    raise RuntimeError(f"No active latent features satisfy SNR > {SNR_THRESHOLD}. Correlation analysis is stopped.")


## Representative-Sigma Selection

In [ ]:
active_sigma = latent_sigma[:, active_feature_indices]
sigma_corr = np.corrcoef(active_sigma, rowvar=False)
if np.ndim(sigma_corr) == 0:
    sigma_corr = np.array([[1.0]])

def save_heatmap(matrix: np.ndarray, row_labels: list[str], col_labels: list[str], title: str, path_base: Path, *, vmin=-1.0, vmax=1.0, cmap="coolwarm", fmt=".2f", caption: str | None = None) -> None:
    matrix = np.asarray(matrix, dtype=np.float64)
    n_rows, n_cols = matrix.shape
    fig_w = max(7.0, 0.68 * n_cols + 2.8)
    fig_h = max(3.8, 0.38 * n_rows + 1.9)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), constrained_layout=True)
    im = ax.imshow(matrix, vmin=vmin, vmax=vmax, cmap=cmap, aspect="auto")
    ax.set_xticks(np.arange(n_cols), labels=col_labels, rotation=45, ha="right")
    ax.set_yticks(np.arange(n_rows), labels=row_labels)
    ax.set_title(title)
    for r in range(n_rows):
        for c in range(n_cols):
            value = matrix[r, c]
            label = "nan" if not np.isfinite(value) else format(value, fmt)
            ax.text(c, r, label, ha="center", va="center", fontsize=7)
    if caption:
        ax.set_xlabel(caption)
    fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    fig.savefig(path_base.with_suffix(".png"), dpi=240)
    fig.savefig(path_base.with_suffix(".pdf"))
    plt.show()

sigma_labels = [f"sigma_{i}" for i in active_feature_indices]
save_heatmap(
    sigma_corr,
    sigma_labels,
    sigma_labels,
    "Active sigma-sigma Pearson correlation",
    OUTPUT_DIR / "heatmaps" / "active_sigma_correlation",
)

if SIGMA_FEATURE_INDEX is not None:
    representative_sigma_index = int(SIGMA_FEATURE_INDEX)
    if representative_sigma_index < 0 or representative_sigma_index >= latent_dim:
        raise ValueError(f"SIGMA_FEATURE_INDEX={representative_sigma_index} is outside [0, {latent_dim}).")
    if representative_sigma_index not in set(active_feature_indices.tolist()):
        raise ValueError(f"SIGMA_FEATURE_INDEX={representative_sigma_index} is not SNR-active. Active indices: {active_feature_indices.tolist()}")
    sigma_selection_method = "user-specified"
else:
    sigma_selection_method = "automatic medoid by active sigma-sigma Pearson correlation"
    if len(active_feature_indices) == 1:
        representative_sigma_index = int(active_feature_indices[0])
    else:
        abs_corr = np.abs(sigma_corr).copy()
        np.fill_diagonal(abs_corr, np.nan)
        representativeness = np.nanmean(abs_corr, axis=1)
        best_score = np.nanmax(representativeness)
        candidate_positions = np.flatnonzero(np.isclose(representativeness, best_score, rtol=0.0, atol=1e-15))
        representative_sigma_index = int(active_feature_indices[candidate_positions[np.argmin(active_feature_indices[candidate_positions])]])

rep_pos = int(np.where(active_feature_indices == representative_sigma_index)[0][0])
if len(active_feature_indices) == 1:
    rep_abs_corrs = np.array([1.0])
    representative_sigma_score = 1.0
else:
    rep_abs_corrs = np.delete(np.abs(sigma_corr[rep_pos]), rep_pos)
    representative_sigma_score = float(np.mean(rep_abs_corrs))

representative_sigma = latent_sigma[:, representative_sigma_index]
sigma_summary = {
    "representative_sigma_index": int(representative_sigma_index),
    "selection_method": sigma_selection_method,
    "mean_abs_corr": float(np.mean(rep_abs_corrs)),
    "min_abs_corr": float(np.min(rep_abs_corrs)),
    "median_abs_corr": float(np.median(rep_abs_corrs)),
    "selection_score": float(representative_sigma_score),
}
pd.DataFrame([sigma_summary]).to_csv(OUTPUT_DIR / "tables" / "representative_sigma_summary.csv", index=False)
pd.DataFrame(sigma_corr, index=sigma_labels, columns=sigma_labels).to_csv(OUTPUT_DIR / "tables" / "active_sigma_correlation.csv")

if len(active_feature_indices) > 1:
    offdiag = np.abs(sigma_corr[~np.eye(sigma_corr.shape[0], dtype=bool)])
    if np.nanmedian(offdiag) < 0.8:
        warnings.warn("Active sigma dimensions are not all strongly correlated; still retaining one representative sigma by the configured criterion.")

print("representative sigma latent index:", representative_sigma_index)
print("selection method                 :", sigma_selection_method)
print("mean abs corr to other sigmas   :", sigma_summary["mean_abs_corr"])
print("min abs corr to other sigmas    :", sigma_summary["min_abs_corr"])
print("median abs corr to other sigmas :", sigma_summary["median_abs_corr"])


## Observable Function Definitions

In [ ]:
if NUMBA_AVAILABLE:
    @njit
    def _e2_e3_beta1_numba(z, eta, phi, jet_radius):
        n = z.shape[0]
        e2 = 0.0
        e3 = 0.0
        for i in range(n - 1):
            for j in range(i + 1, n):
                dij = math.sqrt((eta[i] - eta[j]) ** 2 + (phi[i] - phi[j]) ** 2)
                e2 += z[i] * z[j] * (dij / jet_radius)
        for i in range(n - 2):
            for j in range(i + 1, n - 1):
                dij = math.sqrt((eta[i] - eta[j]) ** 2 + (phi[i] - phi[j]) ** 2)
                for k in range(j + 1, n):
                    dik = math.sqrt((eta[i] - eta[k]) ** 2 + (phi[i] - phi[k]) ** 2)
                    djk = math.sqrt((eta[j] - eta[k]) ** 2 + (phi[j] - phi[k]) ** 2)
                    e3 += z[i] * z[j] * z[k] * (dij * dik * djk / (jet_radius ** 3))
        return e2, e3
else:
    _e2_e3_beta1_numba = None

def _e2_e3_beta1_python(z: np.ndarray, eta: np.ndarray, phi: np.ndarray, jet_radius: float) -> tuple[float, float]:
    n = len(z)
    e2 = 0.0
    e3 = 0.0
    for i in range(n - 1):
        for j in range(i + 1, n):
            dij = math.hypot(float(eta[i] - eta[j]), float(phi[i] - phi[j]))
            e2 += float(z[i] * z[j]) * (dij / jet_radius)
    for i in range(n - 2):
        for j in range(i + 1, n - 1):
            dij = math.hypot(float(eta[i] - eta[j]), float(phi[i] - phi[j]))
            for k in range(j + 1, n):
                dik = math.hypot(float(eta[i] - eta[k]), float(phi[i] - phi[k]))
                djk = math.hypot(float(eta[j] - eta[k]), float(phi[j] - phi[k]))
                e3 += float(z[i] * z[j] * z[k]) * (dij * dik * djk / (jet_radius ** 3))
    return e2, e3

def e2_e3_beta1(z: np.ndarray, eta: np.ndarray, phi: np.ndarray, jet_radius: float) -> tuple[float, float]:
    if len(z) < 2:
        return 0.0, 0.0
    if NUMBA_AVAILABLE:
        return tuple(float(x) for x in _e2_e3_beta1_numba(z.astype(np.float64), eta.astype(np.float64), phi.astype(np.float64), float(jet_radius)))
    return _e2_e3_beta1_python(z, eta, phi, jet_radius)

def prepare_probability_image(image: np.ndarray) -> tuple[np.ndarray, dict[str, float | bool]]:
    arr = np.asarray(image, dtype=np.float64)
    if arr.ndim == 3:
        if arr.shape[0] != 1:
            raise ValueError(f"Expected a single-channel image, got shape {arr.shape}.")
        arr = arr[0]
    if arr.ndim != 2:
        raise ValueError(f"Expected a 2D image after channel selection, got shape {arr.shape}.")
    negative_sum = float(np.abs(arr[arr < 0.0]).sum())
    arr = np.clip(arr, 0.0, None)
    raw_sum = float(arr.sum())
    if raw_sum <= 0.0 or not np.isfinite(raw_sum):
        return arr, {"is_valid": False, "raw_sum": raw_sum, "negative_sum": negative_sum, "normalization_deviation": np.nan}
    z = arr / raw_sum
    return z, {"is_valid": True, "raw_sum": raw_sum, "negative_sum": negative_sum, "normalization_deviation": abs(float(z.sum()) - 1.0)}

def weighted_two_axis_kmeans(eta: np.ndarray, phi: np.ndarray, weights: np.ndarray, max_iter: int = 30) -> np.ndarray:
    coords = np.column_stack([eta, phi]).astype(np.float64)
    weights = weights.astype(np.float64)
    if len(weights) == 1:
        return np.vstack([coords[0], coords[0]])
    first = int(np.argmax(weights))
    dist2_first = np.sum((coords - coords[first]) ** 2, axis=1)
    second_scores = weights * dist2_first
    second_scores[first] = -1.0
    second = int(np.argmax(second_scores))
    axes = np.vstack([coords[first], coords[second]])
    for _ in range(max_iter):
        d2 = np.stack([np.sum((coords - axes[0]) ** 2, axis=1), np.sum((coords - axes[1]) ** 2, axis=1)], axis=1)
        labels = np.argmin(d2, axis=1)
        new_axes = axes.copy()
        for k in range(2):
            mask = labels == k
            if np.any(mask):
                new_axes[k] = np.average(coords[mask], axis=0, weights=weights[mask])
            else:
                other = 1 - k
                farthest = int(np.argmax(weights * np.sum((coords - axes[other]) ** 2, axis=1)))
                new_axes[k] = coords[farthest]
        if np.allclose(new_axes, axes, rtol=0.0, atol=1e-12):
            axes = new_axes
            break
        axes = new_axes
    order = np.lexsort((axes[:, 1], axes[:, 0]))
    return axes[order]

def calculate_observables(image: np.ndarray, eta_grid: np.ndarray, phi_grid: np.ndarray, jet_radius: float) -> dict[str, float]:
    """Calculate image-based jet observables from a single-channel pT-normalized image.

    Pixel centers are treated as massless pseudo-particles with pT fractions z_a.
    The input image is clipped to non-negative float64 values and re-normalized event-by-event.
    """
    z_img, prep = prepare_probability_image(image)
    if not prep["is_valid"]:
        return {"is_valid": False, "raw_sum": prep["raw_sum"], "normalization_deviation": np.nan}
    z_all = z_img.ravel()
    eta_all = eta_grid.ravel().astype(np.float64)
    phi_all = phi_grid.ravel().astype(np.float64)
    positive = z_all > 0.0
    z_pos = z_all[positive]
    eta_pos = eta_all[positive]
    phi_pos = phi_all[positive]

    eta_bar = float(np.sum(z_pos * eta_pos))
    phi_bar = float(np.sum(z_pos * phi_pos))
    delta_eta = eta_pos - eta_bar
    delta_phi = phi_pos - phi_bar
    r = np.sqrt(delta_eta ** 2 + delta_phi ** 2)

    e_tilde = float(np.sum(z_pos * np.cosh(eta_pos)))
    px_tilde = float(np.sum(z_pos * np.cos(phi_pos)))
    py_tilde = float(np.sum(z_pos * np.sin(phi_pos)))
    pz_tilde = float(np.sum(z_pos * np.sinh(eta_pos)))
    m2_tilde = max(e_tilde ** 2 - px_tilde ** 2 - py_tilde ** 2 - pz_tilde ** 2, 0.0)
    pt2_tilde = px_tilde ** 2 + py_tilde ** 2
    rho = m2_tilde / max(pt2_tilde * jet_radius ** 2, np.finfo(np.float64).tiny)
    log_rho = math.log(rho + RHO_EPS)

    lha = float(np.sum(z_pos * np.sqrt(np.maximum(r / jet_radius, 0.0))))
    girth = float(np.sum(z_pos * (r / jet_radius)))
    order = np.argsort(r, kind="mergesort")
    sorted_r = r[order]
    cumulative = np.cumsum(z_pos[order])
    r50 = float(sorted_r[np.searchsorted(cumulative, 0.5, side="left")])
    r80 = float(sorted_r[np.searchsorted(cumulative, 0.8, side="left")])

    active = z_pos > PIXEL_ACTIVE_THRESHOLD
    z_active = z_pos[active]
    eta_active = eta_pos[active]
    phi_active = phi_pos[active]
    n_active = int(len(z_active))
    e2_beta1, e3_beta1 = e2_e3_beta1(z_active, eta_active, phi_active, jet_radius)
    d2_beta1 = e3_beta1 / (e2_beta1 ** 3 + D2_EPS)

    tau1_axis = np.array([eta_bar, phi_bar], dtype=np.float64)
    tau1 = float(np.sum(z_active * np.sqrt((eta_active - tau1_axis[0]) ** 2 + (phi_active - tau1_axis[1]) ** 2)) / jet_radius) if n_active else 0.0
    axes2 = weighted_two_axis_kmeans(eta_active, phi_active, z_active) if n_active else np.zeros((2, 2), dtype=np.float64)
    if n_active:
        d_axis0 = np.sqrt((eta_active - axes2[0, 0]) ** 2 + (phi_active - axes2[0, 1]) ** 2)
        d_axis1 = np.sqrt((eta_active - axes2[1, 0]) ** 2 + (phi_active - axes2[1, 1]) ** 2)
        tau2 = float(np.sum(z_active * np.minimum(d_axis0, d_axis1)) / jet_radius)
    else:
        tau2 = 0.0
    tau21_beta1 = tau2 / (tau1 + TAU_EPS)

    c_eta_eta = float(np.sum(z_pos * delta_eta ** 2))
    c_phi_phi = float(np.sum(z_pos * delta_phi ** 2))
    c_eta_phi = float(np.sum(z_pos * delta_eta * delta_phi))
    trace = c_eta_eta + c_phi_phi
    diff = c_eta_eta - c_phi_phi
    discr = math.sqrt(max(diff ** 2 + 4.0 * c_eta_phi ** 2, 0.0))
    lambda1 = 0.5 * (trace + discr)
    lambda2 = 0.5 * (trace - discr)
    eccentricity = (lambda1 - lambda2) / (lambda1 + lambda2 + COVARIANCE_EPS)
    eccentricity = float(np.clip(eccentricity, 0.0, 1.0))
    principal_axis_angle = 0.5 * math.atan2(2.0 * c_eta_phi, c_eta_eta - c_phi_phi)
    if principal_axis_angle >= math.pi / 2:
        principal_axis_angle -= math.pi
    if principal_axis_angle < -math.pi / 2:
        principal_axis_angle += math.pi
    cos_2alpha = math.cos(2.0 * principal_axis_angle)
    sin_2alpha = math.sin(2.0 * principal_axis_angle)

    effective_pixel_multiplicity = 1.0 / max(float(np.sum(z_pos ** 2)), np.finfo(np.float64).tiny)
    image_entropy = float(-np.sum(z_pos * np.log(z_pos)))

    small_angle_m2_over_pt2 = 0.0
    if n_active >= 2:
        for i in range(n_active - 1):
            for j in range(i + 1, n_active):
                dij2 = (eta_active[i] - eta_active[j]) ** 2 + (phi_active[i] - phi_active[j]) ** 2
                small_angle_m2_over_pt2 += float(z_active[i] * z_active[j] * dij2)

    return {
        "is_valid": True,
        "raw_sum": float(prep["raw_sum"]),
        "normalization_deviation": float(prep["normalization_deviation"]),
        "negative_sum_clipped": float(prep["negative_sum"]),
        "rho": float(rho),
        "log_rho": float(log_rho),
        "LHA": lha,
        "girth": girth,
        "R50_over_R": r50 / jet_radius,
        "R80_over_R": r80 / jet_radius,
        "e2_beta1": float(e2_beta1),
        "e3_beta1": float(e3_beta1),
        "D2_beta1": float(d2_beta1),
        "tau21_beta1": float(tau21_beta1),
        "eccentricity": eccentricity,
        "principal_axis_angle": float(principal_axis_angle),
        "cos_2alpha": float(cos_2alpha),
        "sin_2alpha": float(sin_2alpha),
        "effective_pixel_multiplicity": float(effective_pixel_multiplicity),
        "image_entropy": float(image_entropy),
        "n_active_pixels": n_active,
        "angle_quality_mask": bool(eccentricity >= ANGLE_ECCENTRICITY_CUT),
        "m2_over_pt2_exact": float(m2_tilde / max(pt2_tilde, np.finfo(np.float64).tiny)),
        "small_angle_m2_over_pt2": float(small_angle_m2_over_pt2),
        "tau_axis_prescription": TAU_AXIS_PRESCRIPTION,
    }

print("Numba available:", NUMBA_AVAILABLE)
print("tau21 axis prescription:", TAU_AXIS_PRESCRIPTION)


## Toy-Image Validation

In [ ]:
def blank_image() -> np.ndarray:
    return np.zeros((1, img_size, img_size), dtype=np.float64)

center_phi = int(np.argmin(np.abs(phi_centers)))
center_eta = int(np.argmin(np.abs(eta_centers)))

def normalized_toy(points: list[tuple[int, int, float]]) -> np.ndarray:
    im = blank_image()
    for phi_idx, eta_idx, weight in points:
        im[0, phi_idx, eta_idx] += weight
    im /= im.sum()
    return im

single = normalized_toy([(center_phi, center_eta, 1.0)])
two_eta = normalized_toy([(center_phi, max(center_eta - 4, 0), 1.0), (center_phi, min(center_eta + 4, img_size - 1), 1.0)])
two_phi = normalized_toy([(max(center_phi - 4, 0), center_eta, 1.0), (min(center_phi + 4, img_size - 1), center_eta, 1.0)])
circle = normalized_toy([
    (center_phi, max(center_eta - 4, 0), 1.0),
    (center_phi, min(center_eta + 4, img_size - 1), 1.0),
    (max(center_phi - 4, 0), center_eta, 1.0),
    (min(center_phi + 4, img_size - 1), center_eta, 1.0),
])
two_prong = normalized_toy([(max(center_phi - 5, 0), max(center_eta - 5, 0), 1.0), (min(center_phi + 5, img_size - 1), min(center_eta + 5, img_size - 1), 1.0)])

obs_single = calculate_observables(single, ETA_GRID, PHI_GRID, JET_RADIUS)
obs_two_eta = calculate_observables(two_eta, ETA_GRID, PHI_GRID, JET_RADIUS)
obs_two_phi = calculate_observables(two_phi, ETA_GRID, PHI_GRID, JET_RADIUS)
obs_circle = calculate_observables(circle, ETA_GRID, PHI_GRID, JET_RADIUS)
obs_two_prong = calculate_observables(two_prong, ETA_GRID, PHI_GRID, JET_RADIUS)

assert obs_single["rho"] < 1e-10
assert obs_single["R50_over_R"] < 1e-10
assert abs(obs_single["effective_pixel_multiplicity"] - 1.0) < 1e-10
assert abs(obs_single["image_entropy"]) < 1e-10
assert abs(obs_two_eta["effective_pixel_multiplicity"] - 2.0) < 1e-10
assert abs(obs_two_eta["image_entropy"] - math.log(2.0)) < 1e-10
assert obs_two_eta["eccentricity"] > 0.5
assert obs_two_phi["eccentricity"] > 0.5
assert obs_circle["eccentricity"] < 0.1
assert obs_two_prong["tau21_beta1"] <= obs_two_eta["tau21_beta1"] + 1e-9

toy_summary = pd.DataFrame([
    {"toy": "single_center", **obs_single},
    {"toy": "two_eta", **obs_two_eta},
    {"toy": "two_phi", **obs_two_phi},
    {"toy": "circle", **obs_circle},
    {"toy": "two_prong", **obs_two_prong},
])
display(toy_summary[["toy", "rho", "R50_over_R", "effective_pixel_multiplicity", "image_entropy", "eccentricity", "principal_axis_angle", "cos_2alpha", "sin_2alpha", "tau21_beta1"]])
print("Toy-image observable assertions passed.")


## Full Observable Calculation

In [ ]:
observable_cache_csv = OUTPUT_DIR / "cache" / "observables_raw.csv"
observable_cache_json = OUTPUT_DIR / "cache" / "observables_metadata.json"

if observable_cache_csv.exists() and metadata_matches(observable_cache_json, cache_metadata):
    observables_raw_df = pd.read_csv(observable_cache_csv)
    print("Loaded raw observables from cache:", observable_cache_csv)
else:
    rows = []
    invalid_rows = []
    for local_pos, event_index in tqdm(list(enumerate(analysis_event_indices)), desc="Observable calculation"):
        img = make_event_image(events[int(event_index)], pid_lookup, pid_min, n_channels, img_size, checkpoint)
        obs = calculate_observables(img, ETA_GRID, PHI_GRID, JET_RADIUS)
        obs["event_index"] = int(event_index)
        obs["local_position"] = int(local_pos)
        if not obs.get("is_valid", False):
            invalid_rows.append(obs)
        rows.append(obs)
    observables_raw_df = pd.DataFrame(rows)
    observables_raw_df.to_csv(observable_cache_csv, index=False)
    observable_cache_json.write_text(json.dumps(cache_metadata, indent=2, default=json_ready), encoding="utf-8")
    if invalid_rows:
        pd.DataFrame(invalid_rows).to_csv(OUTPUT_DIR / "tables" / "invalid_events.csv", index=False)
    print("Saved raw observables to cache:", observable_cache_csv)

valid_mask = observables_raw_df["is_valid"].astype(bool).to_numpy()
valid_positions = observables_raw_df.loc[valid_mask, "local_position"].astype(int).to_numpy()
invalid_event_count = int((~valid_mask).sum())
print("valid events  :", int(valid_mask.sum()))
print("invalid events:", invalid_event_count)


## Numerical and Physics Validation

In [ ]:
OBSERVABLE_NAMES = [
    "log_rho",
    "LHA",
    "girth",
    "R50_over_R",
    "R80_over_R",
    "e2_beta1",
    "D2_beta1",
    "tau21_beta1",
    "eccentricity",
    "principal_axis_angle",
    "effective_pixel_multiplicity",
    "image_entropy",
]
ORDINARY_SCALAR_OBSERVABLES = [name for name in OBSERVABLE_NAMES if name != "principal_axis_angle"]

valid_obs = observables_raw_df.loc[valid_mask].copy()
max_norm_deviation = float(valid_obs["normalization_deviation"].max()) if len(valid_obs) else np.nan
print("max image normalization deviation:", max_norm_deviation)

validation_rows = []
for name in OBSERVABLE_NAMES:
    values = pd.to_numeric(valid_obs[name], errors="coerce")
    validation_rows.append({
        "name": name,
        "n_nan": int(values.isna().sum()),
        "n_inf": int(np.isinf(values.to_numpy(dtype=float, na_value=np.nan)).sum()),
        "min": float(np.nanmin(values)) if values.notna().any() else np.nan,
        "median": float(np.nanmedian(values)) if values.notna().any() else np.nan,
        "max": float(np.nanmax(values)) if values.notna().any() else np.nan,
    })
validation_df = pd.DataFrame(validation_rows)
display(validation_df)
validation_df.to_csv(OUTPUT_DIR / "tables" / "observable_validation_summary.csv", index=False)

assert (valid_obs["R50_over_R"] <= valid_obs["R80_over_R"] + 1e-12).all()
assert ((valid_obs["eccentricity"] >= -1e-12) & (valid_obs["eccentricity"] <= 1.0 + 1e-12)).all()
assert ((valid_obs["effective_pixel_multiplicity"] >= 1.0 - 1e-9) & (valid_obs["effective_pixel_multiplicity"] <= valid_obs["n_active_pixels"] + 1e-9)).all()
entropy_upper = np.log(valid_obs["n_active_pixels"].clip(lower=1))
assert ((valid_obs["image_entropy"] >= -1e-9) & (valid_obs["image_entropy"] <= entropy_upper + 1e-9)).all()

exact = valid_obs["m2_over_pt2_exact"].to_numpy(dtype=np.float64)
approx = valid_obs["small_angle_m2_over_pt2"].to_numpy(dtype=np.float64)
rel_diff = (exact - approx) / np.maximum(np.abs(exact), 1e-15)
print("mass cross-check relative diff percentiles:", np.nanpercentile(rel_diff, [0, 5, 50, 95, 100]))
fig, ax = plt.subplots(figsize=(5.4, 4.8), constrained_layout=True)
ax.scatter(approx, exact, s=5, alpha=0.35)
lims = [float(np.nanmin([approx.min(), exact.min()])), float(np.nanmax([approx.max(), exact.max()]))]
ax.plot(lims, lims, color="tab:red", linewidth=1.0)
ax.set_xlabel("small-angle sum z_i z_j DeltaR_ij^2")
ax.set_ylabel("exact image m2 / pT2")
ax.set_title("Jet-mass small-angle cross-check")
fig.savefig(OUTPUT_DIR / "heatmaps" / "mass_small_angle_crosscheck.png", dpi=220)
fig.savefig(OUTPUT_DIR / "heatmaps" / "mass_small_angle_crosscheck.pdf")
plt.show()

nearly_circular_fraction = float((valid_obs["eccentricity"] < ANGLE_ECCENTRICITY_CUT).mean())
print("fraction nearly circular:", nearly_circular_fraction)
print("raw angle range        :", (float(valid_obs["principal_axis_angle"].min()), float(valid_obs["principal_axis_angle"].max())))
print("cos/sin finite        :", np.isfinite(valid_obs[["cos_2alpha", "sin_2alpha"]].to_numpy()).all())
print("active selection uses strict SNR >", SNR_THRESHOLD)

if len(active_feature_indices) > 1:
    offdiag_abs = np.abs(sigma_corr[~np.eye(sigma_corr.shape[0], dtype=bool)])
    print("sigma mean offdiag abs corr  :", float(np.nanmean(offdiag_abs)))
    print("sigma median offdiag abs corr:", float(np.nanmedian(offdiag_abs)))
    print("sigma min offdiag abs corr   :", float(np.nanmin(offdiag_abs)))
else:
    print("only one active sigma; sigma-sigma redundancy matrix is 1x1")


## Build and Save Event-by-Event DataFrame

In [ ]:
latent_valid_mu = latent_mu[valid_positions]
#representative_sigma_valid = representative_sigma[valid_positions]
representative_logvar_valid = 2.0 * np.log(representative_sigma[valid_positions])
feature_df = pd.DataFrame({f"mu_{int(i)}": latent_valid_mu[:, int(i)] for i in active_feature_indices})
#feature_df[f"sigma_{representative_sigma_index}"] = representative_sigma_valid
feature_df[f"logvar_{representative_sigma_index}"] = representative_logvar_valid


base_columns = [
    "event_index",
    "rho",
    "log_rho",
    "LHA",
    "girth",
    "R50_over_R",
    "R80_over_R",
    "e2_beta1",
    "D2_beta1",
    "tau21_beta1",
    "eccentricity",
    "principal_axis_angle",
    "cos_2alpha",
    "sin_2alpha",
    "effective_pixel_multiplicity",
    "image_entropy",
    "n_active_pixels",
    "angle_quality_mask",
]
analysis_df = pd.concat([valid_obs[base_columns].reset_index(drop=True), feature_df.reset_index(drop=True)], axis=1)

latent_observables_csv = OUTPUT_DIR / "tables" / "latent_observables.csv"
analysis_df.to_csv(latent_observables_csv, index=False)
try:
    latent_observables_parquet = OUTPUT_DIR / "tables" / "latent_observables.parquet"
    analysis_df.to_parquet(latent_observables_parquet, index=False)
    print("Saved parquet:", latent_observables_parquet)
except Exception as exc:
    warnings.warn(f"Could not save parquet; CSV was saved normally. Reason: {exc}")

CORRELATION_FEATURE_NAMES = [f"mu_{int(i)}" for i in active_feature_indices] + [f"logvar_{representative_sigma_index}"]
print("saved CSV:", latent_observables_csv)
print("analysis rows:", len(analysis_df))
print("correlation rows:", CORRELATION_FEATURE_NAMES)


## Active `mu` plus Representative `sigma` Pearson Heatmap

In [ ]:
def corr_matrix(df: pd.DataFrame, row_names: list[str], col_names: list[str], method: str) -> pd.DataFrame:
    out = pd.DataFrame(index=row_names, columns=col_names, dtype=float)
    for row in row_names:
        for col in col_names:
            pair = df[[row, col]].replace([np.inf, -np.inf], np.nan).dropna()
            out.loc[row, col] = pair[row].corr(pair[col], method=method) if len(pair) >= 3 else np.nan
    return out

pearson_corr = corr_matrix(analysis_df, CORRELATION_FEATURE_NAMES, OBSERVABLE_NAMES, method="pearson")
pearson_corr.to_csv(OUTPUT_DIR / "tables" / "pearson_active_mu_plus_sigma.csv")
save_heatmap(
    pearson_corr.to_numpy(dtype=float),
    pearson_corr.index.tolist(),
    pearson_corr.columns.tolist(),
    "Pearson correlations: active mu plus representative sigma",
    OUTPUT_DIR / "heatmaps" / "pearson_active_mu_plus_sigma",
    caption="principal_axis_angle is a raw-angle diagnostic column",
)


## Active `mu` plus Representative `sigma` Spearman Heatmap

In [ ]:
spearman_corr = corr_matrix(analysis_df, CORRELATION_FEATURE_NAMES, OBSERVABLE_NAMES, method="spearman")
spearman_corr.to_csv(OUTPUT_DIR / "tables" / "spearman_active_mu_plus_sigma.csv")
save_heatmap(
    spearman_corr.to_numpy(dtype=float),
    spearman_corr.index.tolist(),
    spearman_corr.columns.tolist(),
    "Spearman correlations: active mu plus representative sigma",
    OUTPUT_DIR / "heatmaps" / "spearman_active_mu_plus_sigma",
    caption="rho -> log(rho) is monotonic, so Spearman is unchanged except for ties/numerics",
)


## Principal-Axis Circular Analysis

In [ ]:
def pearson_np(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    x = x[mask]
    y = y[mask]
    if np.std(x) == 0.0 or np.std(y) == 0.0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def circular_linear_corr(feature: np.ndarray, cos_theta: np.ndarray, sin_theta: np.ndarray) -> float:
    rxc = pearson_np(feature, cos_theta)
    rxs = pearson_np(feature, sin_theta)
    rcs = pearson_np(cos_theta, sin_theta)
    denom = max(1.0 - rcs ** 2, np.finfo(np.float64).tiny)
    value = (rxc ** 2 + rxs ** 2 - 2.0 * rxc * rxs * rcs) / denom
    return float(math.sqrt(max(value, 0.0)))

def orientation_table(df: pd.DataFrame, mask: np.ndarray, suffix: str) -> pd.DataFrame:
    sub = df.loc[mask]
    rows = []
    for feature in CORRELATION_FEATURE_NAMES:
        x = sub[feature].to_numpy(dtype=np.float64)
        rows.append({
            "feature": feature,
            f"raw_angle_pearson_diag_{suffix}": pearson_np(x, sub["principal_axis_angle"].to_numpy(dtype=np.float64)),
            f"cos_2alpha_pearson_{suffix}": pearson_np(x, sub["cos_2alpha"].to_numpy(dtype=np.float64)),
            f"sin_2alpha_pearson_{suffix}": pearson_np(x, sub["sin_2alpha"].to_numpy(dtype=np.float64)),
            f"circular_linear_magnitude_{suffix}": circular_linear_corr(x, sub["cos_2alpha"].to_numpy(dtype=np.float64), sub["sin_2alpha"].to_numpy(dtype=np.float64)),
        })
    return pd.DataFrame(rows).set_index("feature")

all_finite_mask = np.isfinite(analysis_df[CORRELATION_FEATURE_NAMES + ["principal_axis_angle", "cos_2alpha", "sin_2alpha"]].to_numpy(dtype=float)).all(axis=1)
angle_quality = analysis_df["angle_quality_mask"].astype(bool).to_numpy()
orientation_all = orientation_table(analysis_df, all_finite_mask, "all")
orientation_cut = orientation_table(analysis_df, all_finite_mask & angle_quality, "eccentricity_cut")
orientation_corr = pd.concat([orientation_all, orientation_cut], axis=1)
orientation_corr.to_csv(OUTPUT_DIR / "tables" / "principal_axis_correlations.csv")
save_heatmap(
    orientation_corr.to_numpy(dtype=float),
    orientation_corr.index.tolist(),
    orientation_corr.columns.tolist(),
    "Principal-axis orientation correlations",
    OUTPUT_DIR / "heatmaps" / "principal_axis_correlations",
    caption="Raw-angle Pearson is diagnostic; doubled-angle circular-linear magnitude is the robust orientation metric.",
)


## Unified 12-Observable Summary

In [ ]:
summary_corr = pearson_corr.copy()
summary_corr["principal_axis_angle"] = orientation_corr["circular_linear_magnitude_eccentricity_cut"]
summary_corr.to_csv(OUTPUT_DIR / "tables" / "correlation_summary_12.csv")
summary_labels = OBSERVABLE_NAMES.copy()
summary_labels[summary_labels.index("principal_axis_angle")] = "principal_axis_angle\n(unsigned circ-lin |r|)"
save_heatmap(
    summary_corr[OBSERVABLE_NAMES].to_numpy(dtype=float),
    summary_corr.index.tolist(),
    summary_labels,
    "Unified 12-observable correlation summary",
    OUTPUT_DIR / "heatmaps" / "correlation_summary_12",
    caption="All entries are signed Pearson correlations except principal-axis orientation, which is an unsigned doubled-angle circular-linear magnitude with the eccentricity quality cut.",
)


# Correlation PCA of active posterior means only.

In [ ]:
# Correlation PCA of active posterior means only.
# This standardizes the active mu features first, so PCA diagonalizes their
# correlation matrix rather than their raw covariance matrix.

ACTIVE_MU_FEATURE_NAMES = [f"mu_{int(i)}" for i in active_feature_indices]

active_mu_df = (
    analysis_df[ACTIVE_MU_FEATURE_NAMES]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

if active_mu_df.shape[1] < 2:
    raise RuntimeError(
        f"Correlation PCA needs at least two active mu features; got {active_mu_df.shape[1]}."
    )

active_mu_mean = active_mu_df.mean(axis=0)
active_mu_std = active_mu_df.std(axis=0, ddof=0)

if (active_mu_std <= 0).any():
    bad = active_mu_std.index[active_mu_std <= 0].tolist()
    raise RuntimeError(f"Cannot run correlation PCA; zero-variance active mu features: {bad}")

active_mu_standardized = (active_mu_df - active_mu_mean) / active_mu_std

mu_corr_for_pca = active_mu_standardized.corr(method="pearson").to_numpy(dtype=np.float64)

# eigh is appropriate because the correlation matrix is symmetric.
eigenvalues, eigenvectors = np.linalg.eigh(mu_corr_for_pca)

# Sort PCs by descending eigenvalue.
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

# Fix arbitrary eigenvector signs deterministically:
# make the largest-absolute-loading entry positive for each PC.
for j in range(eigenvectors.shape[1]):
    pivot = np.argmax(np.abs(eigenvectors[:, j]))
    if eigenvectors[pivot, j] < 0:
        eigenvectors[:, j] *= -1.0

pc_names = [f"mu_PC{k + 1}" for k in range(len(eigenvalues))]
pc_scores = active_mu_standardized.to_numpy(dtype=np.float64) @ eigenvectors
pc_df = pd.DataFrame(pc_scores, index=active_mu_df.index, columns=pc_names)

explained_fraction = eigenvalues / np.sum(eigenvalues)
cumulative_explained_fraction = np.cumsum(explained_fraction)


# Participation-ratio effective dimension of the active-mu PCA spectrum.
# General form:
#   d_eff = (sum_i lambda_i)^2 / sum_i lambda_i^2
# For a correlation matrix, sum_i lambda_i = N_active_mu, so this is also:
#   d_eff = N_active_mu^2 / sum_i lambda_i^2
pca_effective_dimension = (
    np.sum(eigenvalues) ** 2
    / np.sum(eigenvalues ** 2)
)

pca_effective_dimension_from_fraction = 1.0 / np.sum(explained_fraction ** 2)

print("Active-mu PCA effective dimension:")
print(f"  participation ratio       = {pca_effective_dimension:.4f}")
print(f"  from explained fractions  = {pca_effective_dimension_from_fraction:.4f}")
print(f"  number of active mu dims  = {len(eigenvalues)}")


pca_eigen_df = pd.DataFrame({
    "principal_component": pc_names,
    "eigenvalue": eigenvalues,
    "explained_fraction": explained_fraction,
    "cumulative_explained_fraction": cumulative_explained_fraction,
    "pca_effective_dimension": pca_effective_dimension,
})
pca_eigen_df.to_csv(
    OUTPUT_DIR / "tables" / "active_mu_correlation_pca_eigenvalues.csv",
    index=False,
)

pca_loading_df = pd.DataFrame(
    eigenvectors,
    index=ACTIVE_MU_FEATURE_NAMES,
    columns=pc_names,
)
pca_loading_df.to_csv(
    OUTPUT_DIR / "tables" / "active_mu_correlation_pca_loadings.csv"
)

display(pca_eigen_df)

# Eigenvalue bar plot, sorted from large to small.
fig, ax = plt.subplots(figsize=(9.5, 4.6), constrained_layout=True)
x = np.arange(1, len(eigenvalues) + 1)
ax.bar(x, eigenvalues, color="tab:blue", edgecolor="black", linewidth=0.5)
ax.axhline(1.0, color="tab:red", linestyle="--", linewidth=1.1, label="eigenvalue = 1")
ax.set_xlabel("principal component index, sorted by eigenvalue")
ax.set_ylabel("eigenvalue of active-mu correlation matrix")
ax.set_title("Correlation PCA eigenvalues for active posterior means")
ax.set_xticks(x)
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig.savefig(OUTPUT_DIR / "heatmaps" / "active_mu_correlation_pca_eigenvalues.png", dpi=240)
fig.savefig(OUTPUT_DIR / "heatmaps" / "active_mu_correlation_pca_eigenvalues.pdf")
plt.show()

# Correlate the PCA components with the same 12 physical observables.
# principal_axis_angle remains the raw-angle diagnostic column here.
pca_analysis_df = pd.concat(
    [
        pc_df.reset_index(drop=True),
        analysis_df.loc[active_mu_df.index, OBSERVABLE_NAMES].reset_index(drop=True),
    ],
    axis=1,
)

pc_observable_pearson = corr_matrix(
    pca_analysis_df,
    pc_names,
    OBSERVABLE_NAMES,
    method="pearson",
)
pc_observable_pearson.to_csv(
    OUTPUT_DIR / "tables" / "active_mu_pc_observable_pearson.csv"
)

save_heatmap(
    pc_observable_pearson.to_numpy(dtype=float),
    pc_observable_pearson.index.tolist(),
    pc_observable_pearson.columns.tolist(),
    "Pearson correlations: active-mu PCA components vs observables",
    OUTPUT_DIR / "heatmaps" / "active_mu_pc_observable_pearson",
    caption="PCA is performed only on active mu features after standardization; sigma is not included. principal_axis_angle is a raw-angle diagnostic column.",
)

print("Active mu features used for PCA:", ACTIVE_MU_FEATURE_NAMES)
print("Saved eigenvalues:", OUTPUT_DIR / "tables" / "active_mu_correlation_pca_eigenvalues.csv")
print("Saved loadings   :", OUTPUT_DIR / "tables" / "active_mu_correlation_pca_loadings.csv")
print("Saved PC-observable Pearson matrix:", OUTPUT_DIR / "tables" / "active_mu_pc_observable_pearson.csv")

## Observable-Observable Correlations

In [ ]:
observable_pearson = analysis_df[ORDINARY_SCALAR_OBSERVABLES].corr(method="pearson")
observable_spearman = analysis_df[ORDINARY_SCALAR_OBSERVABLES].corr(method="spearman")
observable_pearson.to_csv(OUTPUT_DIR / "tables" / "observable_pearson.csv")
observable_spearman.to_csv(OUTPUT_DIR / "tables" / "observable_spearman.csv")
save_heatmap(
    observable_pearson.to_numpy(dtype=float),
    observable_pearson.index.tolist(),
    observable_pearson.columns.tolist(),
    "Observable-observable Pearson correlations",
    OUTPUT_DIR / "heatmaps" / "observable_pearson",
)
save_heatmap(
    observable_spearman.to_numpy(dtype=float),
    observable_spearman.index.tolist(),
    observable_spearman.columns.tolist(),
    "Observable-observable Spearman correlations",
    OUTPUT_DIR / "heatmaps" / "observable_spearman",
)

pairs_to_check = [
    ("log_rho", "e2_beta1"),
    ("LHA", "girth"),
    ("R50_over_R", "R80_over_R"),
    ("effective_pixel_multiplicity", "image_entropy"),
    ("D2_beta1", "tau21_beta1"),
]
for a, b in pairs_to_check:
    print(f"{a} vs {b}: Pearson={observable_pearson.loc[a, b]:.4f}, Spearman={observable_spearman.loc[a, b]:.4f}")


## Scatter/Hexbin Diagnostics

In [ ]:
def safe_name(name: str) -> str:
    return name.replace("/", "_").replace(" ", "_").replace("\n", "_")

def plot_pair(df: pd.DataFrame, feature: str, observable: str, path: Path, *, show: bool) -> None:
    pair = df[[feature, observable]].replace([np.inf, -np.inf], np.nan).dropna()
    pearson = pair[feature].corr(pair[observable], method="pearson") if len(pair) >= 3 else np.nan
    spearman = pair[feature].corr(pair[observable], method="spearman") if len(pair) >= 3 else np.nan
    fig, ax = plt.subplots(figsize=(5.4, 4.6), constrained_layout=True)
    if len(pair) > 5000:
        hb = ax.hexbin(pair[observable], pair[feature], gridsize=55, mincnt=1, bins="log", cmap="viridis")
        fig.colorbar(hb, ax=ax, label="log10(count)")
    else:
        ax.scatter(pair[observable], pair[feature], s=7, alpha=0.35)
    ax.set_xlabel(observable)
    ax.set_ylabel(feature)
    ax.set_title(f"{feature} vs {observable}")
    ax.text(0.03, 0.97, f"Pearson = {pearson:.3f}\nSpearman = {spearman:.3f}", transform=ax.transAxes, va="top", ha="left", bbox={"facecolor": "white", "alpha": 0.75, "edgecolor": "none"})
    fig.savefig(path.with_suffix(".png"), dpi=220)
    fig.savefig(path.with_suffix(".pdf"))
    if show:
        plt.show()
    else:
        plt.close(fig)

scatter_records = []
shown = 0
for feature in CORRELATION_FEATURE_NAMES:
    top_obs = pearson_corr.loc[feature, ORDINARY_SCALAR_OBSERVABLES].abs().sort_values(ascending=False).head(3).index.tolist()
    for obs_name in top_obs:
        scatter_records.append({"feature": feature, "observable": obs_name, "abs_pearson": float(abs(pearson_corr.loc[feature, obs_name]))})
        show = shown < N_DISPLAY_SCATTER_EXAMPLES
        plot_pair(analysis_df, feature, obs_name, OUTPUT_DIR / "scatter_plots" / f"{safe_name(feature)}__{safe_name(obs_name)}", show=show)
        shown += int(show)
    for obs_name in ["cos_2alpha", "sin_2alpha"]:
        plot_pair(analysis_df, feature, obs_name, OUTPUT_DIR / "scatter_plots" / f"{safe_name(feature)}__{safe_name(obs_name)}", show=False)

scatter_df = pd.DataFrame(scatter_records)
scatter_df.to_csv(OUTPUT_DIR / "tables" / "top_scatter_pairs.csv", index=False)
display(scatter_df.head(12))


## Final Summary

In [ ]:
strongest_pearson = {}
strongest_spearman = {}
for feature in CORRELATION_FEATURE_NAMES:
    strongest_pearson[feature] = pearson_corr.loc[feature, ORDINARY_SCALAR_OBSERVABLES].abs().idxmax()
    strongest_spearman[feature] = spearman_corr.loc[feature, ORDINARY_SCALAR_OBSERVABLES].abs().idxmax()

final_summary = {
    "checkpoint_path": str(CKPT_PATH),
    "data_path": str(DATA_PATH),
    "analyzed_event_count": int(len(analysis_df)),
    "original_event_count": int(len(events)),
    "image_dimensions": (n_channels, img_size, img_size),
    "coordinate_range": {"eta": [-ETA_MAX, ETA_MAX], "phi": [-PHI_MAX, PHI_MAX]},
    "jet_radius": JET_RADIUS,
    "latent_dimension": latent_dim,
    "snr_threshold": SNR_THRESHOLD,
    "active_latent_feature_indices": active_feature_indices.tolist(),
    "active_latent_feature_count": int(len(active_feature_indices)),
    "active_snr": {int(i): float(latent_snr[int(i)]) for i in active_feature_indices},
    "representative_sigma_latent_index": int(representative_sigma_index),
    "representative_sigma_selection_method": sigma_selection_method,
    "representative_sigma_correlation_summary": sigma_summary,
    "invalid_or_removed_events": invalid_event_count,
    "fraction_nearly_circular_events": nearly_circular_fraction,
    "rho_regularization": RHO_EPS,
    "D2_regularization": D2_EPS,
    "tau21_axis_prescription": TAU_AXIS_PRESCRIPTION,
    "output_directory": str(OUTPUT_DIR),
    "strongest_pearson_observable": strongest_pearson,
    "strongest_spearman_observable": strongest_spearman,
}
print(json.dumps(final_summary, indent=2, default=json_ready))
(OUTPUT_DIR / "tables" / "final_summary.json").write_text(json.dumps(final_summary, indent=2, default=json_ready), encoding="utf-8")
